In [1]:
import pandas as pd

In [2]:
basepath = 'C:\\Users\\pnosz\\Documents\\datasets\\sentiment140\\training.1600000.processed.noemoticon.csv'

df = pd.read_csv(basepath, encoding='latin-1', header=None)
df.columns = ['sentiment', 'id', 'date', 'query', 'user', 'tweet']

df = df.sample(frac=1).reset_index(drop=True)

df['sentiment'] = df['sentiment'].replace(4, 1)

df.head()

,sentiment,id,date,query,user,tweet
0,0,1983127373,Sun May 31 12:33:23 PDT 2009,NO_QUERY,Aashay,Not liking how the latest version of #Tweetie ...
1,1,1554017353,Sat Apr 18 15:50:29 PDT 2009,NO_QUERY,Turrislove09,Found new band...parachute! Haha
2,1,1883283984,Fri May 22 08:18:17 PDT 2009,NO_QUERY,tristanx,@jenajean Thank you Jen! You are too kind
3,0,2239673774,Fri Jun 19 08:49:49 PDT 2009,NO_QUERY,dewisekararum,mommy u're soo cruel.. why'd u gave me a reall...
4,1,2182329565,Mon Jun 15 13:01:36 PDT 2009,NO_QUERY,braghaggers,@charliemcdowell http://twitpic.com/7fxh1 - Ha...


In [3]:
df = df[['sentiment', 'tweet']]

df.head()

,sentiment,tweet
0,0,Not liking how the latest version of #Tweetie ...
1,1,Found new band...parachute! Haha
2,1,@jenajean Thank you Jen! You are too kind
3,0,mommy u're soo cruel.. why'd u gave me a reall...
4,1,@charliemcdowell http://twitpic.com/7fxh1 - Ha...


In [4]:
df.to_csv('twitter_data.csv', index=False, encoding='latin-1')

In [5]:
import re
from nltk.corpus import stopwords

In [6]:
stop = set(stopwords.words('english')) - {'no', 'not', 'never', 'nor'}

def preprocess(text):
    text = text.lower()
    text = re.sub(r'<[^>]*>', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'\W+', ' ', text)

    tokens = [word for word in text.split() if word not in stop]

    return ' '.join(tokens)

In [7]:
df_small = pd.read_csv('twitter_data.csv', nrows=100_000, encoding='latin-1')

df_small['tweet'] = df_small['tweet'].apply(preprocess)

X = df_small['tweet'].values
y = df_small['sentiment'].values

df_small.head()

,sentiment,tweet
0,0,not liking latest version tweetie changed high...
1,1,found new band parachute haha
2,1,thank jen kind
3,0,mommy u soo cruel u gave really delicious almo...
4,1,haha awesome


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    stratify=y)

pipeline = Pipeline([
    ('vect', HashingVectorizer(decode_error='ignore',
                               n_features=2**21,
                               ngram_range=(1, 2))),
    ('clf', SGDClassifier())
])

param_grid = [
    {
        'clf__loss': ['log_loss', 'hinge'],
        'clf__alpha': [1e-3, 1e-4, 1e-5, 1e-6],
        'clf__penalty': ['l1', 'l2', 'elasticnet']
    }
]

grid = GridSearchCV(pipeline, param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train)

print(f'Best params: {grid.best_params_}')
print(f'Best CV score: {grid.best_score_*100:.2f}%')

test_score = grid.score(X_test, y_test)
print(f'Test score: {test_score*100:.2f}%')

results = pd.DataFrame(grid.cv_results_)
results[['params', 'mean_test_score', 'std_test_score']]

Best params: {'clf__alpha': 1e-05, 'clf__loss': 'hinge', 'clf__penalty': 'elasticnet'}
Best CV score: 77.28%
Test score: 77.67%


,params,mean_test_score,std_test_score
0,"{'clf__alpha': 0.001, 'clf__loss': 'log_loss',...",0.627188,0.003217
1,"{'clf__alpha': 0.001, 'clf__loss': 'log_loss',...",0.722125,0.004981
2,"{'clf__alpha': 0.001, 'clf__loss': 'log_loss',...",0.692088,0.005498
3,"{'clf__alpha': 0.001, 'clf__loss': 'hinge', 'c...",0.649150,0.004547
4,"{'clf__alpha': 0.001, 'clf__loss': 'hinge', 'c...",0.721113,0.004272
5,"{'clf__alpha': 0.001, 'clf__loss': 'hinge', 'c...",0.701725,0.004159
6,"{'clf__alpha': 0.0001, 'clf__loss': 'log_loss'...",0.726375,0.005659
7,"{'clf__alpha': 0.0001, 'clf__loss': 'log_loss'...",0.751550,0.005480
8,"{'clf__alpha': 0.0001, 'clf__loss': 'log_loss'...",0.744788,0.004668
9,"{'clf__alpha': 0.0001, 'clf__loss': 'hinge', '...",0.736825,0.004883


In [10]:
import pyprind

In [11]:
clf = grid.best_estimator_.named_steps['clf']
vect = grid.best_estimator_.named_steps['vect']
classes = [0, 1]

reader = pd.read_csv('twitter_data.csv',
                     chunksize=10_000,
                     encoding='latin-1')

pbar = pyprind.ProgBar(1_600_000 // 10_000, track_time=True)

prev_chunk = None

for chunk in reader:
    if prev_chunk is not None:
        prev_chunk['tweet'] = prev_chunk['tweet'].apply(preprocess)
        X = vect.transform(prev_chunk['tweet'].values)
        y = prev_chunk['sentiment'].values
    
        clf.partial_fit(X, y, classes=classes)
        pbar.update()
    prev_chunk = chunk

test_chunk = prev_chunk
test_chunk['tweet'] = test_chunk['tweet'].apply(preprocess)
X_test = vect.transform(test_chunk['tweet'].values)
y_test = test_chunk['sentiment'].values

score = clf.score(X_test, y_test)
print(f'Test score on last batch: {score*100:.2f}%')

0% [############################# ] 100% | ETA: 00:00:00

Test score on last batch: 78.82%


In [12]:
clf.partial_fit(X_test, y_test)

,loss,'hinge'
,penalty,'elasticnet'
,alpha,1e-05
,l1_ratio,0.15
,fit_intercept,True
,max_iter,1000
,tol,0.001
,shuffle,True
,verbose,0
,epsilon,0.1
,n_jobs,None


In [13]:
import pickle
import os

In [14]:
dest = os.path.join('twitter_classifier', 'pkl_objects')
if not os.path.exists(dest):
    os.makedirs(dest)

pickle.dump(stop, open(os.path.join(dest, 'stopwords.pkl'), 'wb'), protocol=4)
pickle.dump(clf, open(os.path.join(dest, 'classifier.pkl'), 'wb'), protocol=4)